# Clustering Listwise DPO - Entailment Run + LIPO Training (1000 questions)

Full experiment in one notebook: build **1 good : 4 bad** entailment-labeled preference
lists for `N_QUESTIONS` GSM8K questions, train Mistral-7B with the repo's **LIPO-lambda
listwise trainer**, and evaluate end accuracy on the GSM8K test split.

Good/bad is decided **only by the binarized entailment score** (`THRESHOLD`, default 0.5) -
correctness is only used to pick the reference trace. Mirrors
`generation/code/score_entailment.py` and `generation/code/build_entailment_list.py`.

**Everything persists to Google Drive** and generation is **resume-safe**: if the runtime
disconnects, just reconnect, run the cells top-to-bottom again, and generation continues
where it left off.

Approximate budget on **A100** (~12 compute units/hour):

| Step | Time | Units |
|---|---|---|
| Install + model download | ~5 min | ~1 |
| Generate 1000 q x 10 samples (256 tokens, bf16) | ~4-6 h | ~50-70 |
| Process + entailment scoring (~10k traces) | ~15 min | ~3 |
| Build 1:4 lists | seconds | - |
| LIPO training (2 epochs, LoRA) | ~1-2 h | ~15-25 |
| Eval 200 test questions (greedy, 512 tokens) | ~30-40 min | ~8 |

Runtime: **A100 GPU** (Runtime -> Change runtime type -> A100). Works on T4/L4 too
(auto-falls back to 4-bit), but generation is 3-5x slower.


In [ ]:
# -- Install dependencies ------------------------------------------------------
!nvidia-smi
!pip install -q \
    "transformers==4.45.1" \
    "accelerate==1.13.0" \
    "datasets==4.8.4" \
    peft \
    bitsandbytes \
    sentencepiece \
    tqdm
print('Done.')


In [ ]:
# -- Google Drive mount (persistence) + Hugging Face login ---------------------
from google.colab import drive
drive.mount('/content/drive')

from huggingface_hub import login
login()  # paste your HF token when prompted


In [ ]:
# -- Config --------------------------------------------------------------------
import os, gc, json, re, random
import torch
from tqdm import tqdm

MODEL          = "mistralai/Mistral-7B-v0.1"
NLI_MODEL      = "cross-encoder/nli-deberta-v3-small"
N_QUESTIONS    = 1000  # slice size
N_SAMPLES      = 10    # traces per question
MAX_NEW_TOKENS = 256
SEED           = 42
random.seed(SEED)

# entailment labeling (mirrors build_entailment_list.py)
THRESHOLD = 0.5   # score >= THRESHOLD -> good (1), else bad (0)
N_GOOD    = 1     # good traces per question (each emits its own record)
N_BAD     = 4     # ranked bad traces per record

# training / eval
REPO_URL    = "https://github.com/txshah/clustering-listwise-dpo.git"
REPO_BRANCH = "entailment-1000"
EVAL_LIMIT  = 200   # test questions for the quick eval (full split = 1319, ~3h)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if DEVICE == "cuda" else 0
print(f"Device: {DEVICE}  |  VRAM: {VRAM_GB:.1f} GB" if DEVICE == "cuda" else "No GPU - expect very slow generation")

# all outputs live on Drive so nothing is lost on disconnect
WORK_DIR = "/content/drive/MyDrive/entailment_run_1000"
os.makedirs(WORK_DIR, exist_ok=True)

TAG = f"{N_QUESTIONS}"
QUESTIONS_PATH    = f"{WORK_DIR}/questions_{TAG}.jsonl"
RAW_TRACES_PATH   = f"{WORK_DIR}/raw_traces_{TAG}.jsonl"
PROCESSED_PATH    = f"{WORK_DIR}/processed_{TAG}.jsonl"
SCORED_PATH       = f"{WORK_DIR}/processed_{TAG}_scored.jsonl"
ENTAIL_PAIRS_PATH = f"{WORK_DIR}/entailment_pairs_{TAG}_t{THRESHOLD}.jsonl"
OUTPUT_DIR        = f"{WORK_DIR}/listwise_entailment_{TAG}_t{THRESHOLD}"
EVAL_OUT          = f"{WORK_DIR}/results_listwise_entailment_{TAG}_t{THRESHOLD}.json"

print("Config ready. Work dir:", WORK_DIR)


In [ ]:
# -- Utility functions (mirrors generation/code/utils.py) ----------------------

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

def is_correct(trace, ground_truth):
    """Ground truth appears in the last 300 chars of the trace."""
    return ground_truth.strip() in trace[-300:]

def no_similar(candidate, existing, length_window=500):
    """True when candidate differs by >=length_window chars from every existing trace."""
    for t in existing:
        if abs(len(candidate) - len(t)) < length_window:
            return False
    return True

_ERROR_PHRASES = ["error", "apolog", "i cannot", "i'm unable", "i am unable"]
def exist_error(trace):
    lowered = trace.lower()
    return any(p in lowered for p in _ERROR_PHRASES)

print("Utilities defined.")


## Step 1: Prepare questions (1000-question slice)

In [ ]:
from datasets import load_dataset

def extract_gsm8k_answer(raw_answer):
    m = re.search(r"####\s*([\d,\.]+)", raw_answer)
    if m:
        return m.group(1).replace(",", "").strip()
    return raw_answer.strip()

if os.path.exists(QUESTIONS_PATH):
    print(f"Questions already prepared -> {QUESTIONS_PATH}")
else:
    print("Loading GSM8K train split...")
    dataset = load_dataset("openai/gsm8k", "main", split="train")

    questions = [
        {"idx": i, "question": item["question"], "answer": extract_gsm8k_answer(item["answer"])}
        for i, item in enumerate(dataset)
    ][:N_QUESTIONS]

    save_jsonl(questions, QUESTIONS_PATH)
    print(f"Sliced {len(questions)} questions -> {QUESTIONS_PATH}")


## Step 2: Generate traces (1000 questions x 10 samples) - resume-safe

The slow step (~4-6 h on A100 in bf16). Each question's traces are **appended to Drive
as soon as they are generated**, so a disconnect costs at most one question. Re-running
this cell skips questions that are already done.

Loads the model in bf16 on A100 (>=30 GB VRAM); falls back to 4-bit on T4/L4.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL)

if VRAM_GB >= 30:
    print(f"Loading {MODEL} (bf16) for generation...")
    model_gen = AutoModelForCausalLM.from_pretrained(
        MODEL, torch_dtype=torch.bfloat16, device_map="auto"
    )
else:
    print(f"Loading {MODEL} (4-bit) for generation...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model_gen = AutoModelForCausalLM.from_pretrained(
        MODEL, quantization_config=bnb_config, device_map="auto"
    )
model_gen.eval()

def build_prompt(question, tok):
    if tok.chat_template is not None:
        messages = [{"role": "user", "content": question}]
        return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return f"Question: {question}\nAnswer:"

questions_data = load_jsonl(QUESTIONS_PATH)

# resume: any idx already in the raw file was fully generated (all N_SAMPLES written at once)
done_idxs = set()
if os.path.exists(RAW_TRACES_PATH):
    done_idxs = {r["idx"] for r in load_jsonl(RAW_TRACES_PATH)}
    print(f"Resuming - {len(done_idxs)}/{len(questions_data)} questions already generated.")

todo = [q for q in questions_data if q["idx"] not in done_idxs]

with open(RAW_TRACES_PATH, "a", encoding="utf-8") as out:
    for item in tqdm(todo, desc="generating"):
        prompt = build_prompt(item["question"], tokenizer)
        inputs = tokenizer(prompt, return_tensors="pt").to(model_gen.device)
        prompt_len = inputs["input_ids"].shape[1]

        with torch.no_grad():
            output_ids = model_gen.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=1.0,
                num_return_sequences=N_SAMPLES,
                pad_token_id=tokenizer.eos_token_id,
            )

        for seq in output_ids:
            trace = tokenizer.decode(seq[prompt_len:], skip_special_tokens=True)
            out.write(json.dumps({
                "idx": item["idx"],
                "question": item["question"],
                "answer": item["answer"],
                "trace": trace,
            }, ensure_ascii=False) + "\n")
        out.flush()

print(f"Generation complete -> {RAW_TRACES_PATH}")

# free VRAM before NLI scoring / training
del model_gen
gc.collect()
torch.cuda.empty_cache()


## Step 3: Process traces (sort correct / wrong, dedup disabled)

In [ ]:
from collections import defaultdict

raw = load_jsonl(RAW_TRACES_PATH)
pools = defaultdict(lambda: {"question": "", "answer": "", "correct_solutions": [], "wrong_solutions": []})

for item in tqdm(raw, desc="processing"):
    idx, trace, gt = item["idx"], item["trace"], item["answer"]
    pool = pools[idx]
    pool["question"] = item["question"]
    pool["answer"]   = gt

    if is_correct(trace, gt):
        if not exist_error(trace) and no_similar(trace, pool["correct_solutions"], length_window=0):
            pool["correct_solutions"].append(trace)
    else:
        if no_similar(trace, pool["wrong_solutions"], length_window=0):
            pool["wrong_solutions"].append(trace)

results = [{"idx": idx, **pool} for idx, pool in pools.items()]
save_jsonl(results, PROCESSED_PATH)

n_no_correct = sum(1 for r in results if not r["correct_solutions"])
print(f"Saved {len(results)} questions -> {PROCESSED_PATH}")
print(f"Questions with zero correct traces (will be dropped later): {n_no_correct}")


## Step 4: Entailment scoring (mirrors `score_entailment.py`)

Every trace (correct and wrong) is scored with an NLI cross-encoder against the
**reference** = shortest correct trace of its question. Score = P(entailment).


In [ ]:
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification

# Traces end with the answer, so the tail carries the most signal;
# also keeps inputs within the 512-token limit.
TAIL_CHARS = 800
NLI_BATCH  = 32

print(f"Loading NLI model {NLI_MODEL}...")
nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(DEVICE)
nli_model.eval()

entail_idx = next(
    (i for i, l in nli_model.config.id2label.items() if l.lower() == "entailment"), 2
)

def score_pairs(premises, hypotheses):
    """Return P(entailment) for each (premise, hypothesis) pair."""
    scores = []
    for i in range(0, len(premises), NLI_BATCH):
        enc = nli_tokenizer(
            premises[i : i + NLI_BATCH],
            hypotheses[i : i + NLI_BATCH],
            truncation=True, max_length=512, padding=True, return_tensors="pt",
        ).to(DEVICE)
        with torch.no_grad():
            logits = nli_model(**enc).logits
        probs = F.softmax(logits, dim=-1)
        scores.extend(probs[:, entail_idx].cpu().tolist())
    return scores

scored = []
for item in tqdm(load_jsonl(PROCESSED_PATH), desc="scoring"):
    corrects, wrongs = item["correct_solutions"], item["wrong_solutions"]

    if not corrects:  # no reference -> nothing meaningful to score
        scored.append({**item, "entailment_scores": {"correct": [], "wrong": [0.0] * len(wrongs)}})
        continue

    reference = min(corrects, key=len)
    ref_tail  = reference[-TAIL_CHARS:]

    if len(corrects) == 1:
        correct_scores = [1.0]
    else:
        correct_scores = score_pairs([ref_tail] * len(corrects), [c[-TAIL_CHARS:] for c in corrects])

    wrong_scores = score_pairs([ref_tail] * len(wrongs), [w[-TAIL_CHARS:] for w in wrongs]) if wrongs else []

    scored.append({**item, "entailment_scores": {"correct": correct_scores, "wrong": wrong_scores}})

save_jsonl(scored, SCORED_PATH)
print(f"Saved scored results -> {SCORED_PATH}")

# score distribution - use this to sanity-check THRESHOLD before Step 5
all_scores = [s for r in scored if r["correct_solutions"]
              for s in r["entailment_scores"]["correct"] + r["entailment_scores"]["wrong"]]
print(f"\n{len(all_scores)} scores | min {min(all_scores):.3f} | max {max(all_scores):.3f}")
for lo in [i / 10 for i in range(10)]:
    n = sum(1 for s in all_scores if lo <= s < lo + 0.1)
    print(f"  [{lo:.1f}-{lo + 0.1:.1f}) {'#' * (60 * n // max(1, len(all_scores)))} {n}")


## Step 5: Build 1:4 entailment-labeled lists (mirrors `build_entailment_list.py`)

Correct and wrong pools are **merged**; the only label is `score >= THRESHOLD`.
Output is `ListwiseTrainer`-compatible (`prompt`, `chosen`, `rejected1-4`).

If the skip counts below are high, revisit `THRESHOLD` (in the Config cell) using the
histogram above and re-run from this cell - the output filename carries the threshold,
so runs at different thresholds don't overwrite each other.


In [ ]:
pairs = []
skipped_no_reference = 0
skipped_ratio = 0

for item in tqdm(load_jsonl(SCORED_PATH), desc="building entailment lists"):
    if not item["correct_solutions"]:
        skipped_no_reference += 1
        continue

    es = item["entailment_scores"]
    pooled = list(zip(item["correct_solutions"], es["correct"])) + \
             list(zip(item["wrong_solutions"],   es["wrong"]))

    goods = sorted((p for p in pooled if p[1] >= THRESHOLD), key=lambda x: x[1], reverse=True)
    bads  = sorted((p for p in pooled if p[1] < THRESHOLD),  key=lambda x: x[1], reverse=True)

    if len(goods) < N_GOOD or len(bads) < N_BAD:
        skipped_ratio += 1
        continue

    ranked_bads = bads[:N_BAD]
    for chosen, chosen_score in goods[:N_GOOD]:
        record = {"prompt": item["question"], "chosen": chosen}
        for i, (bad, _) in enumerate(ranked_bads, start=1):
            record[f"rejected{i}"] = bad
        record["chosen_score"]    = chosen_score
        record["rejected_scores"] = [s for _, s in ranked_bads]
        pairs.append(record)

save_jsonl(pairs, ENTAIL_PAIRS_PATH)
print(f"Skipped {skipped_no_reference} questions (no correct solution -> no reference)")
print(f"Skipped {skipped_ratio} questions (fewer than {N_GOOD} good or {N_BAD} bad at threshold {THRESHOLD})")
print(f"Saved {len(pairs)} preference lists ({N_GOOD}:{N_BAD} good:bad) -> {ENTAIL_PAIRS_PATH}")


## Step 6: LIPO-lambda listwise training

Clones the repo (branch `entailment-1000`) and runs `training/train_listwise.py` on the
lists built above. LoRA policy + frozen base as reference (single model load). The
adapter is saved to Drive, so a disconnect after training still keeps the model.
Uses the repo's `listwise_config.yaml` hyperparameters (beta=0.1, lambdas 1.0/0.75/0.5/0.25,
lr 1e-6, effective batch 16, 2 epochs).


In [ ]:
if not os.path.exists("/content/clustering-listwise-dpo"):
    !git clone -b {REPO_BRANCH} {REPO_URL} /content/clustering-listwise-dpo

%cd /content/clustering-listwise-dpo
!python training/train_listwise.py \
    --config training/configs/listwise_config.yaml \
    --dataset_path "$ENTAIL_PAIRS_PATH" \
    --output_dir "$OUTPUT_DIR"
%cd /content
print("Adapter saved ->", OUTPUT_DIR)


## Step 7: Evaluate on GSM8K test split

Greedy decoding, end-accuracy. Starts with `EVAL_LIMIT` questions (default 200,
~30-40 min on A100) for a quick read; set `EVAL_LIMIT = None` in the Config cell and
re-run for the full 1319 (~3 h). For the downstream comparison, evaluate the base model
on the same slice by swapping `--model` for the base model name (commented line).


In [ ]:
limit_flag = f"--limit {EVAL_LIMIT}" if EVAL_LIMIT else ""

%cd /content/clustering-listwise-dpo
!python evaluation/eval_gsm8k.py --model "$OUTPUT_DIR" --output "$EVAL_OUT" $limit_flag
# baseline for comparison (same slice):
# !python evaluation/eval_gsm8k.py --model mistralai/Mistral-7B-v0.1 --output "$WORK_DIR/results_base.json" $limit_flag
%cd /content

print(open(EVAL_OUT).read())


## Notes

- **Disconnected mid-run?** Reconnect, then run all cells top-to-bottom. Steps 1-2 skip
  finished work automatically; Steps 3-5 are cheap to re-run; Step 6 restarts training
  from scratch (it is short enough that mid-training checkpointing is not worth it).
- **Different threshold:** change `THRESHOLD` in the Config cell and re-run Steps 5-7
  only - no regeneration needed. Output files are tagged with the threshold.
- **Everything lives in** `Drive/entailment_run_1000/` - traces, lists, adapter, results.
